# Recurrent Neural Networks (RNN): Long short-term memory (LSTM)

## Table of contents

### [1. Libraries import and Data preparation](#1)
#### [- Droping the 3 weather stations not included in answers](#1.1)
#### [- Removing the 2 types of observations missing multiple entries for most stations](#1.2)
#### [- Filling in 3 individual observations assuming nearby stations have similar weather conditions](#1.3)

### [2. RNN](#2)
#### [2.1 Reshaping the data ](#2.1)
#### [2.2 Spliting the data into training ant testing sets](#2.2)
#### [2.3 Running the model](#2.3)
##### [- 1st iteration](#2.3.1)
##### [- 2nd iteration](#2.3.2)
##### [- 3rd iteration](#2.3.3)
##### [- 4th iteration](#2.3.4)
##### [- 5th iteration](#2.3.5)


## 1. Libraries import and Data preparation
<div id='1'></div>

In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
import tensorflow as tf
from numpy import unique
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Conv1D, Conv2D, Dense, BatchNormalization, Flatten, MaxPooling1D, Dropout
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split



##to ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Creating paths to the data folder of the project and to the folder for saving the charts
path_data = r"C:\Users\dacol\Documents\Data_Project_careerfoundry\ClimateWins_ML\02 Data"

# Importing the unscaled dataframe
df_weather = pd.read_csv(os.path.join(path_data,'Original Data','Dataset-weather-prediction-dataset-processed.csv'), sep =',')

# Importing the rated dataframe for pleasant days
df_rate = pd.read_csv(os.path.join(path_data,'Original Data','Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'), sep =',')

In [5]:
print(df_weather.shape)
print(df_rate.shape)

(22950, 170)
(22950, 16)


In [6]:
df_weather.describe()

,DATE,MONTH,BASEL_cloud_cover,BASEL_wind_speed,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_snow_depth,BASEL_sunshine,...,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_snow_depth,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
count,2.295000e+04,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,...,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000,22950.000000
mean,1.990984e+07,6.509630,5.410763,2.120462,0.758554,1.018013,1.345244,0.222305,0.359564,4.592222,...,5.723355,0.825824,1.014242,1.134490,0.414698,0.001220,3.460989,10.724257,7.901904,13.515752
std,1.813833e+05,3.443672,2.406115,0.732625,0.110699,0.006543,0.931158,0.498995,2.006231,4.310808,...,1.630313,0.071121,0.010727,0.848813,0.844943,0.049383,3.329432,3.328727,3.659393,3.477373
min,1.960010e+07,1.000000,0.000000,0.000000,0.350000,0.974700,0.010000,0.000000,-2.000000,0.000000,...,0.000000,0.380000,0.955100,0.020000,0.000000,0.000000,0.000000,-3.500000,-7.300000,-1.500000
25%,1.975092e+07,4.000000,4.000000,2.100000,0.680000,1.015800,0.540000,0.000000,0.000000,0.400000,...,5.000000,0.790000,1.009400,0.400000,0.010000,0.000000,0.500000,8.800000,6.100000,11.300000
50%,1.991060e+07,7.000000,6.000000,2.100000,0.770000,1.018000,1.130000,0.000000,0.000000,3.600000,...,6.000000,0.820000,1.014200,0.970000,0.280000,0.000000,3.400000,10.700000,7.900000,13.500000
75%,2.007021e+07,9.000000,7.000000,2.100000,0.840000,1.020100,2.070000,0.220000,0.000000,7.900000,...,7.000000,0.870000,1.020900,1.700000,0.410000,0.000000,4.800000,13.000000,10.300000,15.700000
max,2.022103e+07,12.000000,8.000000,16.300000,1.000000,1.045200,4.560000,8.500000,49.000000,16.800000,...,8.000000,1.000000,1.046300,3.980000,90.000000,3.000000,15.800000,23.600000,19.500000,28.400000


#### - Droping the 3 weather stations not included in answers
<div id='1.2'></div>

In [8]:
#isolating the columns names from the rate df since some weather stations are missing
rate_columns = df_rate.columns.to_list()
rate_columns

['DATE',
 'BASEL_pleasant_weather',
 'BELGRADE_pleasant_weather',
 'BUDAPEST_pleasant_weather',
 'DEBILT_pleasant_weather',
 'DUSSELDORF_pleasant_weather',
 'HEATHROW_pleasant_weather',
 'KASSEL_pleasant_weather',
 'LJUBLJANA_pleasant_weather',
 'MAASTRICHT_pleasant_weather',
 'MADRID_pleasant_weather',
 'MUNCHENB_pleasant_weather',
 'OSLO_pleasant_weather',
 'SONNBLICK_pleasant_weather',
 'STOCKHOLM_pleasant_weather',
 'VALENTIA_pleasant_weather']

In [9]:
#isolating the weather columns to check for the stations to trim
weather_columns = df_weather.columns.to_list()
weather_columns

['DATE',
 'MONTH',
 'BASEL_cloud_cover',
 'BASEL_wind_speed',
 'BASEL_humidity',
 'BASEL_pressure',
 'BASEL_global_radiation',
 'BASEL_precipitation',
 'BASEL_snow_depth',
 'BASEL_sunshine',
 'BASEL_temp_mean',
 'BASEL_temp_min',
 'BASEL_temp_max',
 'BELGRADE_cloud_cover',
 'BELGRADE_humidity',
 'BELGRADE_pressure',
 'BELGRADE_global_radiation',
 'BELGRADE_precipitation',
 'BELGRADE_sunshine',
 'BELGRADE_temp_mean',
 'BELGRADE_temp_min',
 'BELGRADE_temp_max',
 'BUDAPEST_cloud_cover',
 'BUDAPEST_humidity',
 'BUDAPEST_pressure',
 'BUDAPEST_global_radiation',
 'BUDAPEST_precipitation',
 'BUDAPEST_sunshine',
 'BUDAPEST_temp_mean',
 'BUDAPEST_temp_min',
 'BUDAPEST_temp_max',
 'DEBILT_cloud_cover',
 'DEBILT_wind_speed',
 'DEBILT_humidity',
 'DEBILT_pressure',
 'DEBILT_global_radiation',
 'DEBILT_precipitation',
 'DEBILT_sunshine',
 'DEBILT_temp_mean',
 'DEBILT_temp_min',
 'DEBILT_temp_max',
 'DUSSELDORF_cloud_cover',
 'DUSSELDORF_wind_speed',
 'DUSSELDORF_humidity',
 'DUSSELDORF_pressure',
 

In [10]:
#splitting the column names to isolate the stations names
answer_stations = [col.split('_')[0] for col in rate_columns if '_' in col ]
answer_stations

['BASEL',
 'BELGRADE',
 'BUDAPEST',
 'DEBILT',
 'DUSSELDORF',
 'HEATHROW',
 'KASSEL',
 'LJUBLJANA',
 'MAASTRICHT',
 'MADRID',
 'MUNCHENB',
 'OSLO',
 'SONNBLICK',
 'STOCKHOLM',
 'VALENTIA']

In [11]:
#keeping only the weather stations maatching the stations from the rates
weather_stations = [col for col in weather_columns if col.split('_')[0] in answer_stations]
weather_stations

['BASEL_cloud_cover',
 'BASEL_wind_speed',
 'BASEL_humidity',
 'BASEL_pressure',
 'BASEL_global_radiation',
 'BASEL_precipitation',
 'BASEL_snow_depth',
 'BASEL_sunshine',
 'BASEL_temp_mean',
 'BASEL_temp_min',
 'BASEL_temp_max',
 'BELGRADE_cloud_cover',
 'BELGRADE_humidity',
 'BELGRADE_pressure',
 'BELGRADE_global_radiation',
 'BELGRADE_precipitation',
 'BELGRADE_sunshine',
 'BELGRADE_temp_mean',
 'BELGRADE_temp_min',
 'BELGRADE_temp_max',
 'BUDAPEST_cloud_cover',
 'BUDAPEST_humidity',
 'BUDAPEST_pressure',
 'BUDAPEST_global_radiation',
 'BUDAPEST_precipitation',
 'BUDAPEST_sunshine',
 'BUDAPEST_temp_mean',
 'BUDAPEST_temp_min',
 'BUDAPEST_temp_max',
 'DEBILT_cloud_cover',
 'DEBILT_wind_speed',
 'DEBILT_humidity',
 'DEBILT_pressure',
 'DEBILT_global_radiation',
 'DEBILT_precipitation',
 'DEBILT_sunshine',
 'DEBILT_temp_mean',
 'DEBILT_temp_min',
 'DEBILT_temp_max',
 'DUSSELDORF_cloud_cover',
 'DUSSELDORF_wind_speed',
 'DUSSELDORF_humidity',
 'DUSSELDORF_pressure',
 'DUSSELDORF_global_

In [12]:
#creating a new df with the desired columns
df_weather_new = df_weather[weather_stations]
df_weather_new.head()

,BASEL_cloud_cover,BASEL_wind_speed,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_snow_depth,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,...,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_snow_depth,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,2.1,0.85,1.018,0.32,0.09,0,0.7,6.5,0.8,...,5,0.88,1.0003,0.45,0.34,0,4.7,8.5,6.0,10.9
1,6,2.1,0.84,1.018,0.36,1.05,0,1.1,6.1,3.3,...,7,0.91,1.0007,0.25,0.84,0,0.7,8.9,5.6,12.1
2,8,2.1,0.90,1.018,0.18,0.30,0,0.0,8.5,5.1,...,7,0.91,1.0096,0.17,0.08,0,0.1,10.5,8.1,12.9
3,3,2.1,0.92,1.018,0.58,0.00,0,4.1,6.3,3.8,...,7,0.86,1.0184,0.13,0.98,0,0.0,7.4,7.3,10.6
4,6,2.1,0.95,1.018,0.65,0.14,0,5.4,3.0,-0.7,...,3,0.80,1.0328,0.46,0.00,0,5.7,5.7,3.0,8.4


#### - Removing the 2 types of observations missing multiple entries for most stations
<div id='1.2'></div>

In [14]:
#Extract the part of the column names after the underscore
column_names = [col.split('_', 1)[1] for col in df_weather.columns if '_' in col]

# Remove duplicates by converting the list to a set and then back to a list
unique_column_names = list(set(column_names))

# Display the list of unique column names
unique_column_names


['temp_min',
 'temp_max',
 'global_radiation',
 'wind_speed',
 'sunshine',
 'precipitation',
 'pressure',
 'humidity',
 'snow_depth',
 'cloud_cover',
 'temp_mean']

In [15]:
# Create a dictionary to store the count of stations for each observation type
station_counts = {}

for obs in unique_column_names:
    # Select columns related to the current observation type
    columns = [col for col in df_weather_new.columns if col.endswith(obs)]
    
    # Count the number of stations (i.e., the number of columns) for the current observation type
    station_counts[obs] = len(columns)

# Print the count of stations for each observation type
print("Number of stations covered by each observation type:")
for obs, count in station_counts.items():
    print(f"{obs}: {count} stations")

Number of stations covered by each observation type:
temp_min: 15 stations
temp_max: 15 stations
global_radiation: 15 stations
wind_speed: 9 stations
sunshine: 15 stations
precipitation: 15 stations
pressure: 14 stations
humidity: 14 stations
snow_depth: 6 stations
cloud_cover: 14 stations
temp_mean: 15 stations


In [16]:
# Identify the two observation types with the minimum counts
observations_to_remove = sorted(station_counts, key=station_counts.get)[:2]
observations_to_remove

['snow_depth', 'wind_speed']

In [17]:
# Matching the columns to remove
columns_to_remove = [col for col in df_weather_new.columns if any(col.endswith(obs) for obs in observations_to_remove)]
columns_to_remove


['BASEL_wind_speed',
 'BASEL_snow_depth',
 'DEBILT_wind_speed',
 'DUSSELDORF_wind_speed',
 'DUSSELDORF_snow_depth',
 'HEATHROW_snow_depth',
 'KASSEL_wind_speed',
 'LJUBLJANA_wind_speed',
 'MAASTRICHT_wind_speed',
 'MADRID_wind_speed',
 'MUNCHENB_snow_depth',
 'OSLO_wind_speed',
 'OSLO_snow_depth',
 'SONNBLICK_wind_speed',
 'VALENTIA_snow_depth']

In [18]:
# Remove these observation types from the dataframe
df_weather_new_cleaned = df_weather_new.drop(columns=columns_to_remove)
df_weather_new_cleaned.head()

,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,BASEL_temp_max,BELGRADE_cloud_cover,...,STOCKHOLM_temp_max,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,10.9,1,...,4.9,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,10.1,6,...,5.0,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,9.9,6,...,4.1,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,10.6,8,...,2.3,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,6.0,8,...,4.3,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


#### - Filling in 3 individual observations assuming nearby stations have similar weather conditions
<div id='1.3'></div>

In [20]:
# Create a list of all unique station names in the dataset

all_stations = set([col.split('_')[0] for col in df_weather_new_cleaned.columns if '_' in col])
all_stations

{'BASEL',
 'BELGRADE',
 'BUDAPEST',
 'DEBILT',
 'DUSSELDORF',
 'HEATHROW',
 'KASSEL',
 'LJUBLJANA',
 'MAASTRICHT',
 'MADRID',
 'MUNCHENB',
 'OSLO',
 'SONNBLICK',
 'STOCKHOLM',
 'VALENTIA'}

In [21]:
# Find the missing stations for the weather observations observations

observation_types = unique_column_names
missing_stations_by_observation = {}

for obs in observation_types:
    # Select columns related to the current observation type
    columns = [col for col in df_weather_new_cleaned.columns if col.endswith(obs)]
    
    # Extract station names by removing the observation type from the column names
    station_names = set([col.replace(f'_{obs}', '') for col in columns])
    
    # Identify stations that are in all_stations but missing from the current observation type
    missing_stations = all_stations - station_names
    
    # Store the missing station names in the dictionary only if exactly one station is missing
    if len(missing_stations) == 1:
        missing_stations_by_observation[obs] = missing_stations

# Print the missing station names for each observation type
for obs, missing_stations in missing_stations_by_observation.items():
    print(f"\nStations missing from {obs}:")
    for station in missing_stations:
        print(station)



Stations missing from pressure:
MUNCHENB

Stations missing from humidity:
STOCKHOLM

Stations missing from cloud_cover:
KASSEL


**Assuming nearby stations have similar weather, we pick one to copy the data from (Ljubljana is near Kassel, Sonnblick is near Munchen, and Oslo is close enough to Stockholm**

In [23]:
# Define the mapping of stations to copy observations from
station_mapping = {
    'KASSEL_cloud_cover': 'LJUBLJANA_cloud_cover',
    'STOCKHOLM_humidity': 'OSLO_humidity',
    'MUNCHENB_pressure': 'SONNBLICK_pressure'
}

# Create the new columns and copy the corresponding weather observations
for target, source in station_mapping.items():
    df_weather_new_cleaned[target] = df_weather_new_cleaned[source]

# Reorder the columns to group observations by weather station
ordered_columns = []
stations = sorted(set(col.split('_')[0] for col in df_weather_new_cleaned.columns))

for station in stations:
    station_columns = [col for col in df_weather_new_cleaned.columns if col.startswith(station)]
    ordered_columns.extend(station_columns)

df_weather_new_cleaned = df_weather_new_cleaned[ordered_columns]

# Display the updated dataframe
df_weather_new_cleaned.head()

,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,BASEL_temp_max,BELGRADE_cloud_cover,...,STOCKHOLM_humidity,VALENTIA_cloud_cover,VALENTIA_humidity,VALENTIA_pressure,VALENTIA_global_radiation,VALENTIA_precipitation,VALENTIA_sunshine,VALENTIA_temp_mean,VALENTIA_temp_min,VALENTIA_temp_max
0,7,0.85,1.018,0.32,0.09,0.7,6.5,0.8,10.9,1,...,0.98,5,0.88,1.0003,0.45,0.34,4.7,8.5,6.0,10.9
1,6,0.84,1.018,0.36,1.05,1.1,6.1,3.3,10.1,6,...,0.62,7,0.91,1.0007,0.25,0.84,0.7,8.9,5.6,12.1
2,8,0.90,1.018,0.18,0.30,0.0,8.5,5.1,9.9,6,...,0.69,7,0.91,1.0096,0.17,0.08,0.1,10.5,8.1,12.9
3,3,0.92,1.018,0.58,0.00,4.1,6.3,3.8,10.6,8,...,0.98,7,0.86,1.0184,0.13,0.98,0.0,7.4,7.3,10.6
4,6,0.95,1.018,0.65,0.14,5.4,3.0,-0.7,6.0,8,...,0.96,3,0.80,1.0328,0.46,0.00,5.7,5.7,3.0,8.4


In [24]:
#Saving the cleaned data set for future reference and use in models

df_weather_new_cleaned.to_pickle(os.path.join(path_data, 'Prepared Data', 'Cleaned_ML_ready.pkl' ))

In [25]:
# Droping the date variable from the rates (pleasant/unpleasant)

df_rate.drop(columns = 'DATE', inplace = True)

In [26]:
# Renaming dataframes as X and y
X, y = df_weather_new_cleaned, df_rate

# Display the shapes of X and y
print(X.shape, y.shape)


(22950, 135) (22950, 15)


## 2. RNN
<div id='2'></div>

### 2.1 Reshaping the data 
<div id='2.1'></div>

In [117]:
# Turn X and y from a df to arrays

X = np.array(X)
y = np.array(y)

In [118]:
# reshaping X to a 3D object
X = X.reshape(-1,15,9)

In [119]:
# Verifying the shape of X

X

array([[[  7.    ,   0.85  ,   1.018 , ...,   6.5   ,   0.8   ,
          10.9   ],
        [  1.    ,   0.81  ,   1.0195, ...,   3.7   ,  -0.9   ,
           7.9   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.4   ,  -0.4   ,
           5.1   ],
        ...,
        [  4.    ,   0.73  ,   1.0304, ...,  -5.9   ,  -8.5   ,
          -3.2   ],
        [  5.    ,   1.0114,   0.05  , ...,   2.2   ,   4.9   ,
           0.98  ],
        [  5.    ,   0.88  ,   1.0003, ...,   8.5   ,   6.    ,
          10.9   ]],

       [[  6.    ,   0.84  ,   1.018 , ...,   6.1   ,   3.3   ,
          10.1   ],
        [  6.    ,   0.84  ,   1.0172, ...,   2.9   ,   2.2   ,
           4.4   ],
        [  4.    ,   0.67  ,   1.017 , ...,   2.3   ,   1.4   ,
           3.1   ],
        ...,
        [  6.    ,   0.97  ,   1.0292, ...,  -9.5   , -10.5   ,
          -8.5   ],
        [  5.    ,   1.0114,   0.05  , ...,   3.    ,   5.    ,
           0.62  ],
        [  7.    ,   0.91  ,   1.0007, ...,   8.

### 2.2 Spliting the data into training ant testing sets
<div id='2.2'></div>

In [121]:
#Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [122]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(16065, 15, 9)
(6885, 15, 9)
(16065, 15)
(6885, 15)


### 2.3 Running the model
<div id='2.3'></div>

In [124]:
# Create a dictionary from the list
stations = {index: station for index, station in enumerate(all_stations)}

# Display the dictionary
stations


{0: 'BUDAPEST',
 1: 'MAASTRICHT',
 2: 'DUSSELDORF',
 3: 'VALENTIA',
 4: 'STOCKHOLM',
 5: 'HEATHROW',
 6: 'MADRID',
 7: 'DEBILT',
 8: 'BASEL',
 9: 'OSLO',
 10: 'SONNBLICK',
 11: 'MUNCHENB',
 12: 'BELGRADE',
 13: 'KASSEL',
 14: 'LJUBLJANA'}

#### - 1st iteration
<div id='2.3.1'></div>

In [126]:
# 1st iteration

epochs = 30
batch_size = 16
n_hidden = 32

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh'))

In [127]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                        │ (None, 32)                  │           5,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 15)                  │             495 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,871 (22.93 KB)

 Trainable params: 5,871 (22.93 KB)

 Non-trainable params: 0 (0.00 B)

In [128]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [129]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.2281 - loss: 24.1800 - val_accuracy: 0.5644 - val_loss: 28.7941
Epoch 2/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.1926 - loss: 24.5173 - val_accuracy: 0.0633 - val_loss: 24.7777
Epoch 3/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.1162 - loss: 23.9388 - val_accuracy: 0.5893 - val_loss: 27.6070
Epoch 4/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1454 - loss: 24.4903 - val_accuracy: 0.1897 - val_loss: 31.9207
Epoch 5/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0924 - loss: 24.5360 - val_accuracy: 0.1069 - val_loss: 30.0269
Epoch 6/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.1229 - loss: 23.1715 - val_accuracy: 0.0353 - val_loss: 28.0652
Epoch 7/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1006 - loss: 23.0441 - val_accuracy: 0.0119 - val_loss: 23.8975
Epoch 8/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1450 - l

The loss keeps being around 24.5, and the accuracy around 0.2, so the model is not converging

In [131]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        BUDAPEST  DEBILT  MAASTRICHT  MADRID  OSLO
True                                                  
BASEL              9       0           0       0     0
BUDAPEST        4230       2         149       1    65
DEBILT            71       0           0       0     0
DUSSELDORF       245       0           0       0     0
HEATHROW          99       0           0       0     0
KASSEL             4       0           0       0     0
LJUBLJANA          3       0           0       0     0
MAASTRICHT      1304       0           1       0     0
MADRID            13       0           0       0     0
MUNCHENB           7       0           0       0     0
OSLO             546       0           0       0     0
SONNBLICK          8       0           0       0     0
STOCKHOLM         32       0           0       0     0
VALENTIA          96       0           0       0     0


#### - 2nd iteration
<div id='2.3.2'></div>

In [133]:
# 2nd iteration - hidden layers up to 64

epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh'))

In [134]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                        │ (None, 64)                  │          18,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 15)                  │             975 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,919 (77.81 KB)

 Trainable params: 19,919 (77.81 KB)

 Non-trainable params: 0 (0.00 B)

In [135]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [136]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.2028 - loss: 24.8147 - val_accuracy: 0.0697 - val_loss: 26.6882
Epoch 2/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0499 - loss: 24.8326 - val_accuracy: 0.0118 - val_loss: 23.2780
Epoch 3/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0381 - loss: 24.5804 - val_accuracy: 0.0110 - val_loss: 25.5025
Epoch 4/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0280 - loss: 25.3163 - val_accuracy: 0.0141 - val_loss: 26.1523
Epoch 5/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0321 - loss: 24.7838 - val_accuracy: 0.0023 - val_loss: 16.3774
Epoch 6/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0462 - loss: 24.8654 - val_accuracy: 0.0029 - val_loss: 27.0348
Epoch 7/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0767 - loss: 25.1296 - val_accuracy: 0.1699 - val_loss: 26.6993
Epoch 8/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0750 - l

The loss is about the same but the accuracy went down to around 0.07

In [138]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        DUSSELDORF  HEATHROW  KASSEL  LJUBLJANA  MAASTRICHT  SONNBLICK  \
True                                                                         
BASEL                0         0       0          9           0          0   
BUDAPEST             1         4       3       4206         231          1   
DEBILT               0         0       0         71           0          0   
DUSSELDORF           0         0       0        245           0          0   
HEATHROW             0         0       0         99           0          0   
KASSEL               0         0       0          4           0          0   
LJUBLJANA            0         0       0          3           0          0   
MAASTRICHT           0         0       0       1304           1          0   
MADRID               0         0       0         13           0          0   
MUNCHENB             0         0       0          7           0          0   
OSLO                 0 

#### - 3rd iteration
<div id='2.3.3'></div>

In [140]:
# 3rd iteration - trying with a different activation : sigmoid

epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

In [141]:
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_7 (LSTM)                        │ (None, 64)                  │          18,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 15)                  │             975 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,919 (77.81 KB)

 Trainable params: 19,919 (77.81 KB)

 Non-trainable params: 0 (0.00 B)

In [142]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [143]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.1025 - loss: 11.0726 - val_accuracy: 0.0434 - val_loss: 9.5271
Epoch 2/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0839 - loss: 11.5585 - val_accuracy: 0.0447 - val_loss: 9.9595
Epoch 3/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0938 - loss: 11.7434 - val_accuracy: 0.0389 - val_loss: 10.3483
Epoch 4/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0884 - loss: 12.1717 - val_accuracy: 0.0346 - val_loss: 11.0780
Epoch 5/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0857 - loss: 12.5720 - val_accuracy: 0.0411 - val_loss: 11.8387
Epoch 6/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0835 - loss: 13.1968 - val_accuracy: 0.0452 - val_loss: 12.4845
Epoch 7/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.0760 - loss: 13.7980 - val_accuracy: 0.0404 - val_loss: 13.1341
Epoch 8/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.0767 - los

The accuracy decreases and the loss increases, so the sigmoid activation isn't performing well either

In [145]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        BUDAPEST  OSLO
True                      
BASEL              9     0
BUDAPEST        4446     1
DEBILT            71     0
DUSSELDORF       245     0
HEATHROW          99     0
KASSEL             4     0
LJUBLJANA          3     0
MAASTRICHT      1305     0
MADRID            13     0
MUNCHENB           7     0
OSLO             545     1
SONNBLICK          8     0
STOCKHOLM         32     0
VALENTIA          96     0


#### - 4th iteration
<div id='2.3.4'></div>

In [193]:
# 4th iteration - low hidden layers seems to work a bit best, let's try out another activation 

epochs = 30
batch_size = 16
n_hidden = 32

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='softmax'))

In [195]:
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                       │ (None, 32)                  │           5,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 15)                  │             495 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,871 (22.93 KB)

 Trainable params: 5,871 (22.93 KB)

 Non-trainable params: 0 (0.00 B)

In [197]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [199]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.0870 - loss: 10.3857 - val_accuracy: 0.0417 - val_loss: 9.1933
Epoch 2/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0873 - loss: 10.7292 - val_accuracy: 0.0376 - val_loss: 9.4062
Epoch 3/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0857 - loss: 10.8789 - val_accuracy: 0.0475 - val_loss: 9.6735
Epoch 4/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0874 - loss: 11.0756 - val_accuracy: 0.0388 - val_loss: 10.2608
Epoch 5/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0881 - loss: 11.0435 - val_accuracy: 0.0324 - val_loss: 10.5949
Epoch 6/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0814 - loss: 11.7127 - val_accuracy: 0.0482 - val_loss: 10.8931
Epoch 7/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0750 - loss: 11.8164 - val_accuracy: 0.0376 - val_loss: 11.1045
Epoch 8/30
1005/1005 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0739 - loss

In [200]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        DUSSELDORF  HEATHROW  LJUBLJANA  MUNCHENB  OSLO  SONNBLICK
True                                                                  
BASEL                0         0          3         2     4          0
BUDAPEST             3         0       2605        89  1748          2
DEBILT               0         0          0         7    64          0
DUSSELDORF           0         0          3        11   231          0
HEATHROW             0         0          5         5    89          0
KASSEL               0         0          0         0     4          0
LJUBLJANA            0         0          1         1     1          0
MAASTRICHT           2         0         75        55  1172          1
MADRID               0         0          0         0    13          0
MUNCHENB             0         1          0         0     6          0
OSLO                 2         1        184        52   306          1
SONNBLICK            1         0    

Loss increases and accuracy decreases, Tanh activation is the best so far.

#### - 5th iteration
<div id='2.3.5'></div>

In [204]:
# 5th iteration - reducing the hidden layers one more time, and increasing both the batch size and the epoch

epochs = 60
batch_size = 32
n_hidden = 16

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh'))

In [206]:
model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_11 (LSTM)                       │ (None, 16)                  │           1,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 15)                  │             255 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,919 (7.50 KB)

 Trainable params: 1,919 (7.50 KB)

 Non-trainable params: 0 (0.00 B)

In [208]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [210]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0475 - loss: 24.9864 - val_accuracy: 0.0020 - val_loss: 31.4921
Epoch 2/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0349 - loss: 23.7801 - val_accuracy: 0.0052 - val_loss: 23.8860
Epoch 3/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0220 - loss: 22.7939 - val_accuracy: 0.0042 - val_loss: 24.5609
Epoch 4/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0090 - loss: 22.6764 - val_accuracy: 1.4524e-04 - val_loss: 27.5374
Epoch 5/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0074 - loss: 21.9223 - val_accuracy: 0.0000e+00 - val_loss: 26.2338
Epoch 6/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0092 - loss: 22.0665 - val_accuracy: 1.4524e-04 - val_loss: 26.4789
Epoch 7/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0106 - loss: 22.4098 - val_accuracy: 0.0013 - val_loss: 24.4627
Epoch 8/60
503/503 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.0178 - loss:

The loss decreases but the accuracy isn't doing better

In [211]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])

# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))

216/216 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Pred        BASEL  BELGRADE  HEATHROW  KASSEL  MUNCHENB  OSLO  SONNBLICK  \
True                                                                       
BASEL           0         0         0       0         9     0          0   
BUDAPEST        3         2       109       6      3391   934          1   
DEBILT          0         0         0       0        71     0          0   
DUSSELDORF      0         0         0       0       245     0          0   
HEATHROW        0         0         0       0        99     0          0   
KASSEL          0         0         0       0         4     0          0   
LJUBLJANA       0         0         0       0         3     0          0   
MAASTRICHT      0         0         0       0      1304     1          0   
MADRID          0         0         0       0        13     0          0   
MUNCHENB        0         0         0       0         7     0          0   
OSLO            0         0         2       0  